# 📊 Week 2: Classical ML Algorithms & Metrics

## Overview
This week covers the ML algorithms that appear in almost every interview:
- Logistic Regression
- Decision Trees
- Random Forests
- Support Vector Machines

## 🎯 Learning Objectives
1. Understand intuition behind each algorithm
2. Know when to use each algorithm
3. Master evaluation metrics
4. Compare models effectively

## ⏱️ Estimated Time: 8-10 hours

---

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# For reproducibility
np.random.seed(42)

print("All imports successful!")

## 1. Create Sample Dataset

We'll create a binary classification dataset to demonstrate all concepts.

In [ ]:
from sklearn.datasets import make_classification

# Create a synthetic classification dataset
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    n_classes=2,
    random_state=42
)

# Create feature names
feature_names = [f'feature_{i}' for i in range(X.shape[1])]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {X.shape[1]}")
print(f"Class distribution: {np.bincount(y)}")

---
## 2. Logistic Regression

### 🧠 Interview Intuition

**What is it?**
- A linear model for binary classification
- Uses sigmoid function to convert linear output to probability

**The Math (simplified):**
$$P(y=1|x) = \sigma(w^T x + b) = \frac{1}{1 + e^{-(w^T x + b)}}$$

**When to use:**
- Baseline model for classification
- When you need interpretable coefficients
- When features have linear relationship with log-odds

**Key hyperparameters:**
- `C`: Regularization strength (smaller = stronger regularization)
- `penalty`: 'l1', 'l2', or 'elasticnet'

In [ ]:
# ============================================================
# LOGISTIC REGRESSION IMPLEMENTATION
# ============================================================

# Scale features (important for logistic regression!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the model
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

# Predictions
y_pred_log = log_reg.predict(X_test_scaled)
y_prob_log = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_log):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_log):.4f}")

# Feature importance (coefficients)
print("\nFeature Coefficients:")
for name, coef in sorted(zip(feature_names, log_reg.coef_[0]), 
                         key=lambda x: abs(x[1]), reverse=True)[:5]:
    print(f"  {name}: {coef:.4f}")

---
## 3. Decision Trees

### 🧠 Interview Intuition

**What is it?**
- Tree-based model that makes decisions by splitting on features
- Splits chosen to maximize information gain (reduce entropy)

**The Math (Gini Impurity):**
$$Gini = 1 - \sum_{i=1}^{C} p_i^2$$

**When to use:**
- Need interpretable model (can visualize the tree)
- Data has non-linear relationships
- Feature interactions are important

**Pros:**
- No scaling required
- Handles non-linear relationships
- Easy to interpret

**Cons:**
- Prone to overfitting
- Unstable (small changes → different tree)

In [ ]:
# ============================================================
# DECISION TREE IMPLEMENTATION
# ============================================================

# Train decision tree (no scaling needed!)
dt = DecisionTreeClassifier(
    max_depth=5,          # Prevent overfitting
    min_samples_split=10, # Minimum samples to split
    min_samples_leaf=5,   # Minimum samples in leaf
    random_state=42
)
dt.fit(X_train, y_train)

# Predictions
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

print("Decision Tree Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_dt):.4f}")

# Feature importance
print("\nFeature Importance:")
for name, imp in sorted(zip(feature_names, dt.feature_importances_), 
                        key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {name}: {imp:.4f}")

In [ ]:
# Demonstrate overfitting with decision trees
print("Demonstrating Overfitting:")
print("="*50)

for depth in [3, 5, 10, None]:
    dt_temp = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt_temp.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, dt_temp.predict(X_train))
    test_acc = accuracy_score(y_test, dt_temp.predict(X_test))
    
    depth_str = str(depth) if depth else "Unlimited"
    print(f"max_depth={depth_str:>10}: Train={train_acc:.4f}, Test={test_acc:.4f}")

---
## 4. Random Forests

### 🧠 Interview Intuition

**What is it?**
- Ensemble of decision trees
- Each tree trained on bootstrap sample
- Each split considers random subset of features

**Why it works:**
- Reduces overfitting through averaging
- Decorrelates trees using random feature selection

**Key hyperparameters:**
- `n_estimators`: Number of trees
- `max_depth`: Maximum depth of each tree
- `max_features`: Features to consider at each split

**Interview favorite question:**
> "What's the difference between bagging and boosting?"

**Bagging (Random Forest):** Train trees in parallel on bootstrap samples, average predictions

**Boosting (XGBoost, etc.):** Train trees sequentially, each correcting previous errors

In [ ]:
# ============================================================
# RANDOM FOREST IMPLEMENTATION
# ============================================================

rf = RandomForestClassifier(
    n_estimators=100,     # Number of trees
    max_depth=10,         # Depth of each tree
    min_samples_split=5,
    max_features='sqrt',  # sqrt(n_features) at each split
    random_state=42,
    n_jobs=-1             # Use all CPU cores
)
rf.fit(X_train, y_train)

# Predictions
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")

# Feature importance
print("\nFeature Importance:")
for name, imp in sorted(zip(feature_names, rf.feature_importances_), 
                        key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {name}: {imp:.4f}")

---
## 5. Support Vector Machines (SVM)

### 🧠 Interview Intuition

**What is it?**
- Finds optimal hyperplane that maximizes margin between classes
- Can use kernel trick for non-linear boundaries

**Key concepts:**
- **Support vectors:** Points closest to decision boundary
- **Margin:** Distance from boundary to nearest points
- **Kernel:** Function to map to higher dimensions

**Common kernels:**
- `linear`: For linearly separable data
- `rbf` (Gaussian): Most versatile, default choice
- `poly`: Polynomial decision boundary

**Key hyperparameters:**
- `C`: Regularization (higher = harder margin)
- `gamma`: RBF kernel width (higher = more complex boundary)

In [ ]:
# ============================================================
# SVM IMPLEMENTATION
# ============================================================

# SVM requires scaled features!
svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=True,  # Enable probability predictions
    random_state=42
)
svm.fit(X_train_scaled, y_train)

# Predictions
y_pred_svm = svm.predict(X_test_scaled)
y_prob_svm = svm.predict_proba(X_test_scaled)[:, 1]

print("SVM Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_svm):.4f}")
print(f"Number of support vectors: {svm.n_support_}")

---
## 6. Evaluation Metrics Deep Dive

### 🎯 The Confusion Matrix

```
                 Predicted
              |  0   |  1   |
         -----+------+------+
Actual   0    |  TN  |  FP  |
         1    |  FN  |  TP  |
```

### Key Metrics:

| Metric | Formula | When to Use |
|--------|---------|-------------|
| Accuracy | (TP+TN)/(TP+TN+FP+FN) | Balanced classes |
| Precision | TP/(TP+FP) | When FP is costly (spam detection) |
| Recall | TP/(TP+FN) | When FN is costly (disease detection) |
| F1 Score | 2*(P*R)/(P+R) | Balance between P and R |
| ROC-AUC | Area under ROC curve | Overall model quality |

In [ ]:
# ============================================================
# CONFUSION MATRIX VISUALIZATION
# ============================================================

def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix'):
    """Plot a confusion matrix with annotations."""
    cm = confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues')
    
    # Add labels
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Negative (0)', 'Positive (1)'])
    ax.set_yticklabels(['Negative (0)', 'Positive (1)'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    
    # Add text annotations
    labels = [['TN', 'FP'], ['FN', 'TP']]
    for i in range(2):
        for j in range(2):
            text = f"{labels[i][j]}\n{cm[i, j]}"
            ax.text(j, i, text, ha='center', va='center', fontsize=14)
    
    plt.colorbar(im)
    plt.tight_layout()
    plt.show()

# Show confusion matrix for best model
plot_confusion_matrix(y_test, y_pred_rf, 'Random Forest Confusion Matrix')

In [ ]:
# ============================================================
# ROC CURVE COMPARISON
# ============================================================

def plot_roc_curves(y_true, predictions_dict):
    """Plot ROC curves for multiple models."""
    plt.figure(figsize=(8, 6))
    
    for name, y_prob in predictions_dict.items():
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc = roc_auc_score(y_true, y_prob)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})')
    
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves Comparison')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Compare all models
predictions = {
    'Logistic Regression': y_prob_log,
    'Decision Tree': y_prob_dt,
    'Random Forest': y_prob_rf,
    'SVM': y_prob_svm
}

plot_roc_curves(y_test, predictions)

---
## 7. Model Comparison Summary

Let's create a comprehensive comparison of all models.

In [ ]:
# ============================================================
# COMPREHENSIVE MODEL COMPARISON
# ============================================================

def evaluate_model(y_true, y_pred, y_prob):
    """Calculate all metrics for a model."""
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }

# Collect results
results = {
    'Logistic Regression': evaluate_model(y_test, y_pred_log, y_prob_log),
    'Decision Tree': evaluate_model(y_test, y_pred_dt, y_prob_dt),
    'Random Forest': evaluate_model(y_test, y_pred_rf, y_prob_rf),
    'SVM': evaluate_model(y_test, y_pred_svm, y_prob_svm)
}

# Create DataFrame for easy comparison
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)

print("Model Comparison:")
print("=" * 70)
print(results_df.to_string())

# Highlight best model per metric
print("\n" + "=" * 70)
print("Best model per metric:")
for col in results_df.columns:
    best_model = results_df[col].idxmax()
    best_score = results_df[col].max()
    print(f"  {col}: {best_model} ({best_score:.4f})")

In [ ]:
# Visualize the comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(results_df.columns))
width = 0.2

for i, (model, scores) in enumerate(results_df.iterrows()):
    ax.bar(x + i * width, scores.values, width, label=model)

ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df.columns, rotation=45, ha='right')
ax.legend(loc='lower right')
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Cross-Validation

**Interview favorite!** Always mention cross-validation when discussing model evaluation.

In [ ]:
# ============================================================
# CROSS-VALIDATION
# ============================================================

print("5-Fold Cross-Validation Results:")
print("=" * 50)

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(random_state=42)
}

cv_results = {}
for name, model in models.items():
    # Use scaled data for LR and SVM
    data = X_train_scaled if name in ['Logistic Regression', 'SVM'] else X_train
    
    scores = cross_val_score(model, data, y_train, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name}:")
    print(f"  Scores: {scores.round(4)}")
    print(f"  Mean: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")
    print()

---
## 📝 Week 2 Interview Questions

### Logistic Regression
1. **Why is it called "regression" if it's for classification?**
   - It regresses to the log-odds of the probability

2. **What's the difference between L1 and L2 regularization?**
   - L1 (Lasso): Sparse solutions, feature selection
   - L2 (Ridge): Shrinks coefficients, handles multicollinearity

### Decision Trees
3. **How do you prevent overfitting in decision trees?**
   - Limit max_depth
   - Set min_samples_split/leaf
   - Prune the tree

4. **Gini vs Entropy?**
   - Gini: Faster, slightly biased toward larger partitions
   - Entropy: Information theory based, more balanced

### Random Forests
5. **Why does random forest work better than a single tree?**
   - Reduces variance through averaging
   - Decorrelated trees via bootstrap + random features

6. **What's Out-of-Bag (OOB) error?**
   - Error estimated on samples not used in each tree's training
   - Free validation without separate test set

### SVM
7. **Explain the kernel trick**
   - Maps data to higher dimension where it becomes linearly separable
   - Computes inner products in high-dim space without explicit transformation

### Metrics
8. **When would you optimize for precision vs recall?**
   - Precision: Spam detection (don't want to miss real emails)
   - Recall: Cancer detection (don't want to miss any cases)

---

## ✅ Week 2 Checklist

- [ ] Explain logistic regression intuitively
- [ ] Understand decision tree splitting criteria
- [ ] Explain bagging vs boosting
- [ ] Know when to use each algorithm
- [ ] Master confusion matrix metrics
- [ ] Explain ROC curves and AUC
- [ ] Perform cross-validation
- [ ] Compare models and explain differences

---

**Next: Week 3 - Data Preprocessing & Feature Engineering** 🚀